# Moxon Tuning Protocol

Use this notebook while iteratively tuning the antenna with a NanoVNA, RigExpert Stick, or similar analyzer. The repository pre-commit/pre-push hook clears notebook cell outputs, so this notebook writes the persistent record to CSV files under `data/`.

Recommended workflow:

1. Measure the antenna using the process in `../TUNING.md`.
2. Select an existing CSV session or create a new one.
3. Fill the form and press **Append Record** after every physical change.
4. Use **Load Session** any time you switch to another CSV.
5. Commit the CSV session file if those measurements should be part of the project history. Keep private field notes uncommitted or in a separate local CSV.


In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import ipywidgets as widgets
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

DATA_DIR = Path('data')
DEFAULT_SESSION = DATA_DIR / 'moxon_tuning_log.csv'
DATA_DIR.mkdir(parents=True, exist_ok=True)

COLUMNS = [
    'timestamp_utc',
    'step',
    'antenna_name',
    'target_frequency_mhz',
    'wire_type',
    'wire_diameter_mm',
    'a_span_mm',
    'b_tail_left_mm',
    'b_tail_right_mm',
    'c_gap_left_mm',
    'c_gap_right_mm',
    'd_tail_left_mm',
    'd_tail_right_mm',
    'e_spacing_mm',
    'feed_gap_mm',
    'outer_pipe_mm',
    'center_pipe_mm',
    'analyzer',
    'calibration_plane',
    'measurement_height_m',
    'orientation',
    'choke_description',
    'coax_route',
    'f_min_swr_mhz',
    'swr_min',
    'target_swr',
    'r_ohm_at_target',
    'x_ohm_at_target',
    'return_loss_db_at_target',
    'front_db',
    'rear_db',
    'front_to_back_db',
    'change_made',
    'notes',
]

TEXT_DEFAULTS = {
    'antenna_name': '2m Moxon prototype',
    'wire_type': 'insulated copper',
    'analyzer': 'NanoVNA',
    'calibration_plane': 'end of analyzer jumper at feed point',
    'orientation': 'horizontal',
    'coax_route': 'away from feed point at right angle',
    'change_made': 'initial build',
}

NUMERIC_DEFAULTS = {
    'step': 0,
    'target_frequency_mhz': 145.0,
}

def sanitize_session_name(name):
    raw = name.strip()
    stem = Path(raw).stem if Path(raw).suffix.lower() == '.csv' else raw
    safe = ''.join(ch if ch.isalnum() or ch in ('-', '_') else '_' for ch in stem)
    return safe or 'moxon_tuning_log'

def session_path(name):
    path = Path(name)
    if path.suffix.lower() != '.csv':
        path = path.with_suffix('.csv')
    if not path.is_absolute():
        path = DATA_DIR / path.name
    return path

def list_sessions():
    sessions = sorted(DATA_DIR.glob('*.csv'))
    if DEFAULT_SESSION not in sessions:
        sessions.insert(0, DEFAULT_SESSION)
    return sessions

def empty_log():
    return pd.DataFrame(columns=COLUMNS)

def load_log(path):
    if path.exists():
        return pd.read_csv(path).reindex(columns=COLUMNS)
    return empty_log()

def save_log(df, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    df = df.reindex(columns=COLUMNS)
    df.to_csv(path, index=False)
    return df

def create_session(name):
    path = session_path(sanitize_session_name(name))
    if not path.exists():
        save_log(empty_log(), path)
    return path

def append_entry(entry, path):
    row = {column: entry.get(column, '') for column in COLUMNS}
    row['timestamp_utc'] = entry.get('timestamp_utc') or datetime.now(timezone.utc).isoformat(timespec='seconds')
    df = load_log(path)
    df = pd.concat([df, pd.DataFrame([row])], ignore_index=True)
    return save_log(df, path)

def numeric_or_blank(value):
    return '' if value is None else value

def front_to_back(front, rear):
    if front is None or rear is None:
        return ''
    return float(front) - float(rear)

def next_step(path):
    df = load_log(path)
    if df.empty:
        return 0
    steps = pd.to_numeric(df['step'], errors='coerce').dropna()
    return int(steps.max()) + 1 if len(steps) else 0

create_session(DEFAULT_SESSION.name)


## Session And Entry Form

Use **New Session Name** to create or restore a separate CSV file, for example `2m_baseline`, `2m_after_new_wire`, or `70cm_vertical`. Session files are stored in `notebooks/data/`.


In [ ]:
session_dropdown = widgets.Dropdown(description='Session CSV', layout=widgets.Layout(width='520px'))
new_session_name = widgets.Text(value='', description='New Session Name', layout=widgets.Layout(width='520px'))
create_button = widgets.Button(description='Create / Restore Session', button_style='info')
load_button = widgets.Button(description='Load Session')
session_status = widgets.HTML()

def refresh_sessions(selected=None):
    sessions = list_sessions()
    options = [(path.name, str(path)) for path in sessions]
    session_dropdown.options = options
    if selected is not None:
        session_dropdown.value = str(selected)
    elif options and session_dropdown.value not in [value for _, value in options]:
        session_dropdown.value = options[0][1]

def current_path():
    return Path(session_dropdown.value)

refresh_sessions(DEFAULT_SESSION)

def make_float(description, value=None, width='250px'):
    return widgets.FloatText(value=value, description=description, layout=widgets.Layout(width=width))

def make_text(description, value='', width='520px'):
    return widgets.Text(value=value, description=description, layout=widgets.Layout(width=width))

fields = {
    'step': widgets.IntText(value=0, description='Step', layout=widgets.Layout(width='250px')),
    'antenna_name': make_text('Antenna', TEXT_DEFAULTS['antenna_name']),
    'target_frequency_mhz': make_float('Target MHz', NUMERIC_DEFAULTS['target_frequency_mhz']),
    'wire_type': make_text('Wire Type', TEXT_DEFAULTS['wire_type']),
    'wire_diameter_mm': make_float('Wire Dia mm'),
    'a_span_mm': make_float('A Span mm'),
    'b_tail_left_mm': make_float('B Left mm'),
    'b_tail_right_mm': make_float('B Right mm'),
    'c_gap_left_mm': make_float('C Left mm'),
    'c_gap_right_mm': make_float('C Right mm'),
    'd_tail_left_mm': make_float('D Left mm'),
    'd_tail_right_mm': make_float('D Right mm'),
    'e_spacing_mm': make_float('E Spacing mm'),
    'feed_gap_mm': make_float('Feed Gap mm'),
    'outer_pipe_mm': make_float('Outer Pipe mm'),
    'center_pipe_mm': make_float('Center Pipe mm'),
    'analyzer': make_text('Analyzer', TEXT_DEFAULTS['analyzer']),
    'calibration_plane': make_text('Calibration', TEXT_DEFAULTS['calibration_plane']),
    'measurement_height_m': make_float('Height m'),
    'orientation': widgets.Dropdown(value='horizontal', options=['horizontal', 'vertical', 'other'], description='Orientation', layout=widgets.Layout(width='250px')),
    'choke_description': make_text('Choke', ''),
    'coax_route': make_text('Coax Route', TEXT_DEFAULTS['coax_route']),
    'f_min_swr_mhz': make_float('Min SWR MHz'),
    'swr_min': make_float('SWR Min'),
    'target_swr': make_float('Target SWR'),
    'r_ohm_at_target': make_float('R Ohm'),
    'x_ohm_at_target': make_float('X Ohm'),
    'return_loss_db_at_target': make_float('Return Loss dB'),
    'front_db': make_float('Front dB'),
    'rear_db': make_float('Rear dB'),
    'change_made': make_text('Change', TEXT_DEFAULTS['change_made']),
    'notes': widgets.Textarea(value='', description='Notes', layout=widgets.Layout(width='760px', height='90px')),
}

append_button = widgets.Button(description='Append Record', button_style='success')
reload_button = widgets.Button(description='Reload Table')
form_status = widgets.HTML()
table_output = widgets.Output()
plot_output = widgets.Output()

def clear_measurement_fields():
    for key in ['f_min_swr_mhz', 'swr_min', 'target_swr', 'r_ohm_at_target', 'x_ohm_at_target', 'return_loss_db_at_target', 'front_db', 'rear_db']:
        fields[key].value = None
    fields['change_made'].value = ''
    fields['notes'].value = ''

def collect_entry():
    entry = {}
    for key, field in fields.items():
        entry[key] = numeric_or_blank(field.value)
    entry['front_to_back_db'] = front_to_back(fields['front_db'].value, fields['rear_db'].value)
    return entry

def render_log():
    path = current_path()
    df = load_log(path)
    with table_output:
        table_output.clear_output()
        display(df.tail(12))
    with plot_output:
        plot_output.clear_output()
        if df.empty:
            print('No records in this session yet.')
            return
        plot_df = df.copy()
        for column in ['step', 'f_min_swr_mhz', 'swr_min', 'target_swr', 'r_ohm_at_target', 'x_ohm_at_target', 'front_to_back_db']:
            plot_df[column] = pd.to_numeric(plot_df[column], errors='coerce')
        columns = [column for column in ['f_min_swr_mhz', 'swr_min', 'target_swr', 'front_to_back_db'] if plot_df[column].notna().any()]
        if not columns:
            print('No numeric measurement columns to plot yet.')
            return
        plot_df.plot(x='step', y=columns, marker='o', subplots=True, figsize=(9, 7))
        plt.tight_layout()
        plt.show()

def load_selected_session(*_):
    path = current_path()
    create_session(path.name)
    fields['step'].value = next_step(path)
    session_status.value = f'<b>Loaded:</b> {path}'
    render_log()

def create_or_restore_session(*_):
    name = new_session_name.value or DEFAULT_SESSION.name
    path = create_session(name)
    refresh_sessions(path)
    load_selected_session()

def append_record(*_):
    path = current_path()
    df = append_entry(collect_entry(), path)
    form_status.value = f'<b>Saved:</b> step {fields["step"].value} to {path.name}'
    fields['step'].value = next_step(path)
    clear_measurement_fields()
    render_log()

create_button.on_click(create_or_restore_session)
load_button.on_click(load_selected_session)
append_button.on_click(append_record)
reload_button.on_click(lambda *_: render_log())
session_dropdown.observe(lambda change: load_selected_session() if change['name'] == 'value' else None, names='value')

geometry_box = widgets.VBox([
    widgets.HTML('<h3>Geometry</h3>'),
    widgets.HBox([fields['a_span_mm'], fields['e_spacing_mm'], fields['feed_gap_mm']]),
    widgets.HBox([fields['b_tail_left_mm'], fields['b_tail_right_mm']]),
    widgets.HBox([fields['c_gap_left_mm'], fields['c_gap_right_mm']]),
    widgets.HBox([fields['d_tail_left_mm'], fields['d_tail_right_mm']]),
    widgets.HBox([fields['outer_pipe_mm'], fields['center_pipe_mm']]),
])
measurement_box = widgets.VBox([
    widgets.HTML('<h3>Analyzer Reading</h3>'),
    widgets.HBox([fields['f_min_swr_mhz'], fields['swr_min'], fields['target_swr']]),
    widgets.HBox([fields['r_ohm_at_target'], fields['x_ohm_at_target'], fields['return_loss_db_at_target']]),
    widgets.HBox([fields['front_db'], fields['rear_db']]),
])
setup_box = widgets.VBox([
    widgets.HTML('<h3>Setup</h3>'),
    widgets.HBox([fields['step'], fields['target_frequency_mhz'], fields['wire_diameter_mm']]),
    fields['antenna_name'],
    fields['wire_type'],
    fields['analyzer'],
    fields['calibration_plane'],
    widgets.HBox([fields['measurement_height_m'], fields['orientation']]),
    fields['choke_description'],
    fields['coax_route'],
])
notes_box = widgets.VBox([fields['change_made'], fields['notes'], widgets.HBox([append_button, reload_button]), form_status])
session_box = widgets.VBox([session_dropdown, widgets.HBox([load_button]), new_session_name, create_button, session_status])

display(widgets.VBox([session_box, setup_box, geometry_box, measurement_box, notes_box, table_output, plot_output]))
load_selected_session()
